## 1. Run cblaster Search for the smoEFGK Gene Cluster


In [ ]:
%%bash
#!/bin/bash
#PBS -N cblaster
#PBS -l nodes=node5:ppn=30,walltime=1000:00:00
#PBS -j oe
#PBS -o /public/home/luyt/Data/Ral/1082_cblaster/smoefgk/cblaster.log
#PBS -q public

cd $PBS_O_WORKDIR

source /public/home/luyt/miniconda3/bin/activate cblaster_new

echo "Job started at: $(date '+%Y-%m-%d %H:%M:%S')"

db="/public/home/luyt/Data/Ral/allral_genome/1266_cblasterdb/1266_allral.dmnd"
query="/public/home/luyt/Data/Ral/1082_cblaster/smoefgk/query_interm7.faa"

cblaster search -m local -db ${db} -qf ${query} -o summary.csv -b binary.csv --session_file session.json -ode "," -ohh -odc 6 -hs 50000 -cp 25

echo "Job finished at: $(date '+%Y-%m-%d %H:%M:%S')"

## 2. Summarize Gene-Cluster Patterns


In [ ]:
import pandas as pd
infile = r"F:\backup_50_public\Ral\allral_smoefgk\analysis\1082_binary_filtered.csv"      # 你的输入文件（csv/tsv都行）
out_prefix = r"F:\backup_50_public\Ral\allral_smoefgk\analysis\1082_pattern_analysis"         # 输出文件前缀
# ============================

# 自动识别分隔符（逗号或tab）
df = pd.read_csv(infile, sep=None, engine="python")

fixed_cols = ["Organism", "Scaffold", "Start", "End", "Score"]
gene_cols = [c for c in df.columns if c not in fixed_cols]

# 基因列转成0/1整数
df[gene_cols] = df[gene_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

# 每行的0/1模式字符串
df["pattern"] = df[gene_cols].astype(str).agg("".join, axis=1)

# 全局统计模式出现次数
pattern_counts = df["pattern"].value_counts()

# 按出现次数从高到低排名（并用pattern字典序做稳定排序）
patterns_sorted = sorted(pattern_counts.index, key=lambda p: (-pattern_counts[p], p))
pattern_to_rank = {p: i + 1 for i, p in enumerate(patterns_sorted)}

# 给每行标注模式排名
df["pattern_rank"] = df["pattern"].map(pattern_to_rank)

# ====== 输出1：模式排名表 ======
pattern_table = pd.DataFrame({
    "pattern": patterns_sorted,
    "count": [int(pattern_counts[p]) for p in patterns_sorted],
    "rank": [pattern_to_rank[p] for p in patterns_sorted],
})

# 把pattern拆回各基因列，方便看哪个基因缺失
bits = pattern_table["pattern"].apply(list).apply(lambda x: list(map(int, x)))
bits_df = pd.DataFrame(bits.tolist(), columns=gene_cols)
pattern_table = pd.concat([pattern_table, bits_df], axis=1)

pattern_table.to_csv(f"{out_prefix}.pattern_rank_table.tsv", sep="\t", index=False)

# ====== 输出2：每个样本出现的模式编号（按行顺序）======
org_rows = (
    df.groupby("Organism")["pattern_rank"]
      .apply(list)
      .reset_index(name="pattern_ranks_in_rows")
)

org_rows["pattern_ranks_in_rows_str"] = org_rows["pattern_ranks_in_rows"].apply(
    lambda x: ",".join(map(str, x))
)

org_rows["pattern_ranks_unique_str"] = org_rows["pattern_ranks_in_rows"].apply(
    lambda x: ",".join(map(str, sorted(set(x))))
)

org_rows.to_csv(f"{out_prefix}.organism_pattern_summary.tsv", sep="\t", index=False)

# ====== 输出3：每行带pattern和rank，方便追踪 ======
df.to_csv(f"{out_prefix}.rows_with_pattern_rank.tsv", sep="\t", index=False)

print("Done!")
print(f"{out_prefix}.pattern_rank_table.tsv")
print(f"{out_prefix}.organism_pattern_summary.tsv")
print(f"{out_prefix}.rows_with_pattern_rank.tsv")

## 3. Retrieve Contig-Length Information

In [ ]:
import csv
import re
from pathlib import Path


ROOT = Path(r"F:\backup_50_public\Ral\allral_smoefgk")
SUMMARY = ROOT / "analysis" / "organism_pattern_summary.tsv"
BINARY = ROOT / "binary.csv"
GBFF_DIR = Path(r"F:\backup_50_public\Ral\allral_genome\1267\1267\file")
OUT_DIR = ROOT / "result710"


def read_table(path, delimiter):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle, delimiter=delimiter))


def gbff_lengths(path):
    lengths = {}
    locus_name = None
    locus_length = None
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("LOCUS"):
                fields = line.split()
                locus_name = fields[1] if len(fields) > 1 else None
                locus_length = int(fields[2]) if len(fields) > 2 and fields[2].isdigit() else None
                if locus_name and locus_length is not None:
                    lengths[locus_name] = locus_length
            elif line.startswith("VERSION"):
                fields = line.split()
                if fields and len(fields) > 1 and locus_length is not None:
                    lengths[fields[1]] = locus_length
    return lengths


summary_rows = read_table(SUMMARY, "\t")
selected = [r for r in summary_rows if r["pattern_ranks_in_rows"] in {"[2, 3]", "[3, 2]"}]
selected_by_id = {r["Organism"]: r for r in selected}
binary_rows = read_table(BINARY, ",")
matched = [r for r in binary_rows if r["Organism"] in selected_by_id]

all_lengths = {}
missing_gbff = []
for organism in sorted(selected_by_id):
    gbff = GBFF_DIR / f"{organism}.gbff"
    if not gbff.exists():
        missing_gbff.append(organism)
        continue
    all_lengths[organism] = gbff_lengths(gbff)

output_rows = []
missing_contigs = []
for row in matched:
    organism = row["Organism"]
    scaffold = row["Scaffold"]
    lookup = all_lengths.get(organism, {})
    length = lookup.get(scaffold)
    if length is None:
        # Defensive fallback for records where NZ_ is present on only one side.
        alternate = scaffold[3:] if scaffold.startswith("NZ_") else "NZ_" + scaffold
        length = lookup.get(alternate)
    status = "matched" if length is not None else "contig_not_found"
    if length is None:
        missing_contigs.append((organism, scaffold))
    enriched = {
        "GCF": organism,
        "pattern": selected_by_id[organism]["pattern_ranks_in_rows"],
        **{k: v for k, v in row.items() if k != "Organism"},
        "Contig_length_bp": "" if length is None else length,
        "Length_lookup_status": status,
        "GBFF_file": f"{organism}.gbff",
    }
    output_rows.append(enriched)

OUT_DIR.mkdir(parents=True, exist_ok=True)
selected_path = OUT_DIR / "selected_115_GCF_samples.tsv"
with selected_path.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=selected[0].keys(), delimiter="\t")
    writer.writeheader()
    writer.writerows(selected)

annotated_path = OUT_DIR / "binary_pattern_2_3_3_2_contig_lengths.csv"
with annotated_path.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=output_rows[0].keys())
    writer.writeheader()
    writer.writerows(output_rows)

pattern_counts = {}
for row in selected:
    p = row["pattern_ranks_in_rows"]
    pattern_counts[p] = pattern_counts.get(p, 0) + 1

unique_matched_samples = {r["Organism"] for r in matched}
report = [
    f"Selected summary rows: {len(selected)}",
    f"Unique selected GCF samples: {len(selected_by_id)}",
    f"Pattern [2, 3] samples: {pattern_counts.get('[2, 3]', 0)}",
    f"Pattern [3, 2] samples: {pattern_counts.get('[3, 2]', 0)}",
    f"Matched binary.csv rows: {len(matched)}",
    f"Unique matched binary.csv samples: {len(unique_matched_samples)}",
    f"Unique matched contigs: {len({(r['Organism'], r['Scaffold']) for r in matched})}",
    f"Missing samples in binary.csv: {len(set(selected_by_id) - unique_matched_samples)}",
    f"Missing GBFF files: {len(missing_gbff)}",
    f"Rows with contig length found: {len(output_rows) - len(missing_contigs)}",
    f"Rows with contig not found in GBFF: {len(missing_contigs)}",
]
if missing_gbff:
    report.append("Missing GBFF sample IDs: " + ", ".join(missing_gbff))
if missing_contigs:
    report.append("Missing contigs: " + ", ".join(f"{o}/{c}" for o, c in missing_contigs))
(OUT_DIR / "contig_length_validation.txt").write_text("\n".join(report) + "\n", encoding="utf-8")

print("\n".join(report))

## 4. Calculate Distances to Contig Edges


In [1]:
# Calculate distances from each gene-cluster region to the nearest contig edge.
from pathlib import Path

import numpy as np
import pandas as pd


input_path = Path(r"E:\Data\Ralstonia\new_analysis\allral_smoefgk\result710\test\binary_intermedia_selected_115_with_contig_info.csv")
output_path = Path(r"E:\Data\Ralstonia\new_analysis\allral_smoefgk\result710\test\binary_intermedia_selected_115_with_contig_info_edge_distance.csv")
report_path = Path(r"E:\Data\Ralstonia\new_analysis\allral_smoefgk\result710\contig_edge_distance_summary.txt")
thresholds = [1000, 2000, 5000, 10000]

df = pd.read_csv(input_path)
required = ["Organism", "Scaffold", "Start", "End", "Contig_length_bp", "Length_lookup_status"]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

for column in ["Start", "End", "Contig_length_bp"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

valid = df["Start"].notna() & df["End"].notna() & df["Contig_length_bp"].notna()
left = pd.Series(np.nan, index=df.index)
right = pd.Series(np.nan, index=df.index)
distance_start = pd.Series(np.nan, index=df.index)
distance_end = pd.Series(np.nan, index=df.index)
nearest_edge = pd.Series("", index=df.index, dtype="object")
nearest_distance = pd.Series(np.nan, index=df.index)
region_fraction = pd.Series(np.nan, index=df.index)

left.loc[valid] = df.loc[valid, ["Start", "End"]].min(axis=1)
right.loc[valid] = df.loc[valid, ["Start", "End"]].max(axis=1)
distance_start.loc[valid] = (left.loc[valid] - 1).clip(lower=0)
distance_end.loc[valid] = (df.loc[valid, "Contig_length_bp"] - right.loc[valid]).clip(lower=0)
nearest_distance.loc[valid] = pd.concat([distance_start, distance_end], axis=1).min(axis=1).loc[valid]
nearest_edge.loc[valid] = np.select(
    [distance_start.loc[valid] == distance_end.loc[valid], distance_start.loc[valid] < distance_end.loc[valid]],
    ["both", "start"],
    default="end",
)
region_fraction.loc[valid] = (
    (right.loc[valid] - left.loc[valid] + 1) / df.loc[valid, "Contig_length_bp"]
).round(6)

df["Region_left_bp"] = left
df["Region_right_bp"] = right
df["Distance_to_contig_start_bp"] = distance_start
df["Distance_to_contig_end_bp"] = distance_end
df["Nearest_contig_edge"] = nearest_edge
df["Nearest_contig_edge_distance_bp"] = nearest_distance
df["Region_fraction_of_contig"] = region_fraction

for threshold in thresholds:
    left_near = valid & (distance_start <= threshold)
    right_near = valid & (distance_end <= threshold)
    df[f"Near_contig_edge_{threshold // 1000}kb"] = np.where(valid, left_near | right_near, "")
    df[f"Edge_side_within_{threshold // 1000}kb"] = np.select(
        [left_near & right_near, left_near, right_near],
        ["both", "start", "end"],
        default="none",
    )
    df.loc[~valid, f"Edge_side_within_{threshold // 1000}kb"] = ""

df.to_csv(output_path, index=False)

report_lines = [
    f"Input rows: {len(df)}",
    f"Rows with valid coordinates and contig length: {int(valid.sum())}",
    f"Rows with invalid or missing coordinates: {int((~valid).sum())}",
    "Coordinate handling: distances are measured from both contig ends.",
    "",
    "Nearest-edge threshold summary:",
]

for threshold in thresholds:
    left_near = valid & (distance_start <= threshold)
    right_near = valid & (distance_end <= threshold)
    report_lines.append(
        f"{threshold} bp: near_any={int((left_near | right_near).sum())}, "
        f"start_only={int((left_near & ~right_near).sum())}, "
        f"end_only={int((right_near & ~left_near).sum())}, "
        f"both={int((left_near & right_near).sum())}, "
        f"none={int((valid & ~left_near & ~right_near).sum())}"
    )

report_lines.extend(["", f"Output CSV: {output_path}"])
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("\n".join(report_lines))


Input rows: 239
Rows with valid coordinates and contig length: 239
Rows with invalid or missing coordinates: 0
Coordinate handling: distances are measured from both contig ends.

Nearest-edge threshold summary:
1000 bp: near_any=130, start_only=67, end_only=63, both=0, none=109
2000 bp: near_any=195, start_only=88, end_only=97, both=10, none=44
5000 bp: near_any=228, start_only=101, end_only=117, both=10, none=11
10000 bp: near_any=229, start_only=100, end_only=115, both=14, none=10

Output CSV: E:\Data\Ralstonia\new_analysis\allral_smoefgk\result710\test\binary_intermedia_selected_115_with_contig_info_edge_distance.csv
